# NB_lab_test_trends

Builds `adb_caresync_project.gold.lab_test_trends` (one row per test type per calendar month) from silver lab_results and calendar, per STTM Silver to Gold v1.0. Includes month-over-month abnormal rate trend via LAG window function. Load type: full rebuild (CREATE OR REPLACE TABLE AS SELECT).

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS adb_caresync_project.gold;

CREATE OR REPLACE TABLE adb_caresync_project.gold.lab_test_trends AS
WITH monthly_agg AS (
  -- Aggregate lab results to test_name + month grain
  SELECT
    lr.test_name,
    YEAR(lr.test_date)                                        AS year,
    MONTH(lr.test_date)                                       AS month,
    COUNT(*)                                                  AS total_tests,
    COUNT_IF(lr.result_status = 'Abnormal')                   AS abnormal_test_count,
    COUNT_IF(lr.result_status = 'Critical')                   AS critical_test_count
  FROM adb_caresync_project.silver.lab_results lr
  GROUP BY lr.test_name, YEAR(lr.test_date), MONTH(lr.test_date)
),
with_rates AS (
  -- Compute rate percentages
  SELECT
    ma.*,
    ROUND(ma.abnormal_test_count * 100.0 / ma.total_tests, 2) AS abnormal_rate_pct,
    ROUND(ma.critical_test_count * 100.0 / ma.total_tests, 2) AS critical_rate_pct
  FROM monthly_agg ma
),
with_window AS (
  -- Add LAG window function for month-over-month comparison
  SELECT
    wr.*,
    LAG(wr.abnormal_rate_pct) OVER (PARTITION BY wr.test_name ORDER BY wr.year, wr.month) AS prev_month_abnormal_rate_pct
  FROM with_rates wr
)
SELECT
  CAST(w.test_name AS STRING)                                                     AS test_name,
  CAST(w.year AS INT)                                                             AS year,
  CAST(w.month AS INT)                                                            AS month,
  CAST(c.month_name AS STRING)                                                    AS month_name,
  CAST(CONCAT('Q', c.quarter) AS STRING)                                          AS quarter_name,
  CAST(w.total_tests AS INT)                                                      AS total_tests,
  CAST(w.abnormal_test_count AS INT)                                              AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                              AS critical_test_count,
  CAST(w.abnormal_rate_pct AS DECIMAL(5,2))                                       AS abnormal_rate_pct,
  CAST(w.critical_rate_pct AS DECIMAL(5,2))                                       AS critical_rate_pct,
  CAST(w.prev_month_abnormal_rate_pct AS DECIMAL(5,2))                            AS prev_month_abnormal_rate_pct,
  CAST(
    ROUND(w.abnormal_rate_pct - w.prev_month_abnormal_rate_pct, 2) AS DECIMAL(6,2)
  )                                                                               AS abnormal_rate_change_pct
FROM with_window w
LEFT JOIN adb_caresync_project.silver.calendar c
  ON c.date_key = CAST(date_format(make_date(w.year, w.month, 1), 'yyyyMMdd') AS INT);

In [0]:
updated_df = spark.table("adb_caresync_project.gold.lab_test_trends")
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))